# Retriever Evaluation Pipeline (Consolidated)

This notebook evaluates the performance of `dense`, `sparse`, `ensemble`, and `hybrid` retrievers.
It uses a simplified Python loop instead of LangGraph.

**Metrics:**
- Hit Rate@TopK
- NDCG@TopK
- Accuracy@Top1
- Relevance Score (LLM Judge 0-5)

In [1]:
import sys
import os
import random
import math
import asyncio
from typing import List, Dict, Any
from tabulate import tabulate
from sqlalchemy import text

# Ensure project root is in sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from app.core.config import settings
from app.db.session import SessionLocal
from app.db.vector_db import (
    get_dense_retriever,
    get_sparse_retriever,
    get_ensemble_retriever,
    get_hybrid_retriever,
    get_dense_multi_query_retriever,
    get_multi_query_ensemble_retriever,
    get_hyde_retriever,
    get_reranker_retriever
)

e:\pythonProject\P-plip-inference\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup & Configuration

In [2]:
def get_llm(model="solar-pro2"):
    # Use solar-pro as configured
    return ChatUpstage(api_key=settings.UPSTAGE_API_KEY, model=model)

## 2. Helper Functions: Data Gen & Metrics

In [3]:
from langchain_core.output_parsers import ListOutputParser, StrOutputParser
# === Data Generation ===

def get_random_attractions(limit: int = 1):
    """
    Fetch random attractions from DB.
    """
    db = SessionLocal()
    try:
        query = text(
            """
            SELECT no, title, overview 
            FROM attractions 
            WHERE overview IS NOT NULL AND overview != '' 
            AND ST_Distance_Sphere(
                POINT(longitude, latitude), 
                POINT(:user_lng, :user_lat)
            ) <= (:radius_km * 1000)
            ORDER BY RAND() 
            LIMIT :limit
            """
        )
        result = db.execute(query, {"limit": limit, "user_lat": 37.5666, "user_lng": 126.9780, "radius_km": 3}).fetchall()
        return [
            {"no": row.no, "title": row.title, "overview": row.overview}
            for row in result
        ]
    finally:
        db.close()

def generate_synthetic_query(category: str, k=5) -> str:
    """
    Generate a search query based on overview using LLM.
    """
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
        [역할]
        당신은 관광지를 찾고싶은 여행객입니다.
        제시된 [카테고리]를 방문하고 싶어하는 사용자들이 던질 법한 질문을 {k}개 생성하세요.
        단, 카테고리별로 세부적인 유형을 나누어서 질문을 생성하세요
        예를 들어 음식점의 경우 카페, 한식집, 양식집, 일식 전문점 등
        쇼핑의 경우 백화점, 전통시장등의 세부 유형을 나누어 질문을 생성하세요.
        
        [지침]
        1. 질문은 친구나 AI에게 말하는 듯한 자연스러운 구어체여야 합니다.
        2. 특정 장소 이름이나 특정 지역명(예: 종로, 시청)은 절대 포함하지 마세요.
        3. 각 질문은 서로 다른 상황(예: 혼자, 가족과, 데이트, 비즈니스 등)을 가정하여 다양하게 만드세요.
        4. 답변은 오직 질문 내용만 출력하며, 각 질문은 줄바꿈으로 구분하세요.      
        5. 절대로 부가적인 설명이나 번호등을 붙이지말고 정확히 5줄이 나오도록 출력하세요.  
        [카테고리]: {category}
        """),
    ])

    # [수정] StrOutputParser를 사용하여 결과를 문자열로 받음
    chain = prompt | llm | StrOutputParser()
    
    print(f"✍️ {category} 관련 가상 질문 생성 중...")
    raw_response = chain.invoke({"category": category, "k": k})
    
    # 줄바꿈으로 쪼개고 빈 줄은 제거하여 리스트 반환
    queries = [q.strip() for q in raw_response.strip().split('\n') if q.strip()]
    return queries[:k] # 정확히

def check_hit_rate(ground_truth_id: int, retrieved_docs: list) -> int:
    for doc in retrieved_docs:
        retrieved_id = doc.metadata.get("no")
        if str(retrieved_id) == str(ground_truth_id):
            return 1
    return 0

def calculate_ndcg(ground_truth_id: int, retrieved_docs: list) -> float:
    for i, doc in enumerate(retrieved_docs):
        retrieved_id = doc.metadata.get("no")
        if str(retrieved_id) == str(ground_truth_id):
            rank = i + 1
            return 1.0 / math.log2(rank + 1)
    return 0.0

def calculate_top1_accuracy(ground_truth_id: int, retrieved_docs: list) -> int:
    if not retrieved_docs:
        return 0
    top_doc = retrieved_docs[0]
    retrieved_id = top_doc.metadata.get("no")
    if str(retrieved_id) == str(ground_truth_id):
        return 1
    return 0

## 3. Main Evaluation Logic

In [4]:
async def run_single_evaluation(query: str, gt_id: int, k: int=5):
    """
    Runs 4 retrievers and evaluates metrics for each.
    Returns a dict of results.
    """
    # 1. Initialize Retrievers
    # Note: VectorStore creation is cached or cheap enough in this context
#    filter = create_geo_radius_filter(lat=37.5666, lon=126.9780, radius_km=3)
    retrievers = {
        "dense": get_dense_retriever(k=k),
        "sparse": get_sparse_retriever(k=k),
        "ensemble": get_ensemble_retriever(k=k),
        "hybrid": get_hybrid_retriever(k=k),
        "multi_query_dense": get_dense_multi_query_retriever(k=k, llm=get_llm()),
        "multi_query_ensemble": get_multi_query_ensemble_retriever(k=k, llm=get_llm()),
    }
    
    results = {}
    
    for name, retriever in retrievers.items():
        # Retrieve (async invoke if possible, else invoke)
        docs = await retriever.ainvoke(query)
        
        # Metrics
        hit = check_hit_rate(gt_id, docs)
        ndcg = calculate_ndcg(gt_id, docs)
        acc = calculate_top1_accuracy(gt_id, docs)
        
        # LLM Judge on Top-1
        if docs:
            top_doc = docs[0]
            content = f"Title: {top_doc.metadata.get('title')}\nOverview: {top_doc.metadata.get('overview')}"
            #eval_res = evaluate_relevance(query, content)
            #score = eval_res.score
        else:
            score = 0
            
        results[name] = {
            "hit_rate": hit,
            "ndcg": ndcg,
            "accuracy": acc,
            #"score": score
        }
        
    return results

## 4. Run Experiment

In [5]:
# import logging

# logging.basicConfig()
# logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [6]:
num_samples = 10
#attractions = get_random_attractions(limit=num_samples)
categories = ["관광지","문화시설","축제공연행사","여행코스","레포츠","숙박","쇼핑","음식점"]
dataset = []

print(f"Generated categories {categories} with {num_samples} samples:")
for category in categories:
    query = generate_synthetic_query(category, num_samples)
    dataset.extend(query)
    print(f"- [{category}] Q: {query}")

print(f"최종 데이터 : {dataset}")

Generated categories ['관광지', '문화시설', '축제공연행사', '여행코스', '레포츠', '숙박', '쇼핑', '음식점'] with 10 samples:
✍️ 관광지 관련 가상 질문 생성 중...
- [관광지] Q: ['혼자 여행 중에 가볼 만한 역사적인 장소가 있을까요?', '가족과 함께 즐기기 좋은 자연 경관이 있는 곳은 어디인가요?', '커플이 산책하기 좋은 분위기 있는 공원이 있을까요?', '비즈니스 미팅 후 잠시 들러서 휴식할 수 있는 관광명소는 어디인가요?', '아이들과 체험 활동을 할 수 있는 문화 유적지가 있을까요?', '야간 조명이 아름다운 관광지는 어디가 있을까요?', '현지인들만 아는 숨은 명소가 있을까요?', '전통 문화를 경험할 수 있는 축제나 행사가 있는 곳은 어디인가요?', '사진 찍기 좋은 랜드마크가 있는 관광지가 있을까요?', '힐링하기 좋은 조용한 사찰이나 유적지가 있을까요?']
✍️ 문화시설 관련 가상 질문 생성 중...
- [문화시설] Q: ['혼자 가기 좋은 조용한 갤러리 어디 있을까요?', '가족 여행 중에 아이들이 체험할 수 있는 박물관 추천해주세요.', '데이트 코스로 분위기 좋은 역사박물관 알고 싶어요.', '비즈니스 미팅 후 동료들과 가볼 만한 전시관 있을까요?', '할머니와 함께 가기 편한 전통공예 체험장 어디인지 궁금해요.', '친구들과 밤새 놀기 좋은 야간 개장 전시공간 있나요?', '혼자 여행 중 지역 문화를 느낄 수 있는 소규모 문화원 알려주세요.', '연인과 사진 찍기 좋은 야외 조각공원 추천해요.', '아이 방학 때 가볼 만한 과학체험관 어디인지 궁금해요.', '부모님과 함께 가기 좋은 전통 공연장 있을까요?']
✍️ 축제공연행사 관련 가상 질문 생성 중...
- [축제공연행사] Q: ['혼자 축제에 가려고 하는데 혼자서도 재밌게 즐길 수 있는 공연이 있을까요?', '가족들과 함께 볼 수 있는 어린이 친화적인 공연 행사가 있을까요?', '커플이 함께 보면 좋을 로맨틱한 분위기의 공

## 평가자 설정

In [7]:
import pandas as pd
import asyncio
from datasets import Dataset
from tabulate import tabulate
from ragas import evaluate
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI
from ragas.metrics import AspectCritic
from app.db.filters import create_geo_radius_filter
from langchain_upstage.embeddings import UpstageEmbeddings
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
filter = create_geo_radius_filter(lat=37.5666, lon=126.9780, radius_km=10)


## 검색 함수

In [8]:

async def fetch_retrieval_results(dataset, k=5, filter=None):
    """
    1단계: 리트리버 초기화 및 병렬 검색 수행
    """
    # 리트리버 초기화
    retrievers = {
        "ensemble": get_ensemble_retriever(k=k, dense_weight=0.8, sparse_weight=0.2, filter=filter),
        "hybrid": get_hybrid_retriever(filter=filter, k=k),
        "hyde": get_hyde_retriever(k=k, llm=get_llm(model="solar-pro2"), filter=filter),
        "dense": get_dense_retriever(k=k, filter=filter),
        "sparse": get_sparse_retriever(k=k, filter=filter)
    }
    retriever_names = list(retrievers.keys())
    ragas_data_collections = {name: [] for name in retriever_names}

    print(f"🚀 총 {len(dataset)}개 샘플 검색 시작 (병렬 처리)...")

    for query in dataset:
        # asyncio.gather를 사용하여 모든 리트리버를 동시에 실행
        tasks = [retrievers[name].ainvoke(query) for name in retriever_names]
        results = await asyncio.gather(*tasks)

        for name, docs in zip(retriever_names, results):
            combined_contexts = [
                f"장소명: {doc.metadata.get('title', 'N/A')}\n내용: {doc.metadata.get('overview', '')}" 
                for doc in docs
            ]
            ragas_data_collections[name].append({
                "user_input": query,
                "retrieved_contexts": combined_contexts
            })
            
    return ragas_data_collections
    

## 평가 함수

In [9]:

# 1. 평가자 설정 (Upstage Solar 모델 및 임베딩)
evaluator_llm = get_llm("solar-pro2") # solar-pro2

# 1. 카테고리 일치성 
category_alignment = AspectCritic(
    name="category_alignment",
    definition="""
    [역할]
    당신은 검색된 장소가 사용자의 '구체적 요구'를 정확히 관통하는지 검증하는 엄격한 감사관입니다.

    [평가 기준 1 - 무관용 원칙]
    1. 상위 범주가 같더라도 하위 범주가 다르면 즉시 0점 처리하십시오. (예: '라멘'을 원하는데 '우동' 집을 추천한 경우)
    2. 질문에 포함된 '특수 목적'이 문서에 명시되어 있지 않다면 0점 처리하십시오. (예: '비건 식당'을 찾는데 '채소 메뉴가 있는 일반 식당'인 경우)
    3. 명칭의 유사성에 속지 마십시오. 이름이 'OO 카페'라 할지라도 문서 내용상 '베이커리'가 주력이고 앉을 자리가 없다면, '대화하기 좋은 카페'를 찾는 질문에는 0점입니다.
    4. 부정 제약 조건이 있다면 반드시 확인하십시오. (예: '프랜차이즈 제외'인데 대형 브랜드인 경우 0점)
    
    [평가 기준 2 - 상호 배타적 분류]
    1. 핵심 정체성 확인: 질문자가 '카페'를 찾으면, 결과는 '커피/차/디저트'가 주력인 공간이어야 합니다. 
    2. 교차 오염 금지 (Zero Tolerance): 
       - '카페' 검색 결과에 '해물', '회', '탕', '구이' 등 식사 메뉴가 주력인 정보가 포함되어 있다면, 이는 '치명적 오검색'으로 간주하고 즉시 0점을 부여하십시오.
       - "식당 내에 카페가 있다"거나 "후식으로 커피를 준다"는 식의 보조적 정보는 무시하십시오. 사용자의 주 목적은 '카페'이지 '해물 식당'이 아닙니다.
    3. 명시적 키워드 대조: 문서의 제목이나 주 설명에 질문의 카테고리(예: 카페)를 부정하는 단어(예: 횟집, 전문점, 매운탕)가 존재하면 뒤도 돌아보지 말고 0점을 주십시오.

    [점수 부여]
    - 1점: 사용자가 정의한 카테고리와 장소의 실질적 주력 서비스가 100% 일치하며, 대체 불가능한 경우.
    - 0점: 범주가 모호하거나, 상위 카테고리만 일치하거나, 사용자의 특정 요구 조건을 문서에서 명시적으로 확인할 수 없는 경우.
    """,
    strictness=2
)

# 2. 분위기 및 조건 관련성 (추상적 요구사항 체크)
vibe_relevance = AspectCritic(
    name="vibe_relevance",
    definition="""
    [역할]
    당신은 '팩트'만 믿는 까칠한 탐정입니다. 문서에 '적혀 있지 않은 내용'을 상상해서 점수를 주는 행위는 당신의 직무 유기입니다.

    [채점 가이드라인: 상상력 금지]
    1. 형용사 검증: '조용한', '힙한', '친절한' 등을 요구했다면, 문서 내에 그 단어 혹은 그에 준하는 객관적 묘사(예: '대화가 안 들릴 정도의 소음', '조명과 음악이 트렌디한')가 반드시 있어야 합니다.
    2. 맥락적 비약 차단: "대형 몰이니까 주차가 편하겠지", "고급 호텔이니까 서비스가 좋겠지" 같은 당신의 상식은 버리십시오. 문서에 '주차장 완비', '친절한 서비스'라는 말이 없으면 0점입니다.
    3. 복합 조건의 파괴: 사용자가 2가지 이상의 조건(예: '조용하고 공부하기 좋은')을 걸었다면, 두 가지가 모두 텍스트로 증명되어야 합니다. 하나라도 없으면 실패한 검색입니다.

    [통과 기준: 1점]
    - 사용자의 요구 사항이 문서의 텍스트를 통해 '객관적으로 입증'되는 경우에만 부여.

    [탈락 기준: 0점]
    - "그럴 법하다"는 생각이 드는 경우 (즉시 탈락).
    - 문서에 단순히 장소 이름과 주소만 있는 경우 (판단 불가로 탈락).
    - 질문의 분위기와 상충하는 단어가 하나라도 섞여 있는 경우.
    """,
    strictness=2
)


In [10]:
from datasets import Dataset
from ragas import evaluate, RunConfig

def evaluate_ragas_data(ragas_data_collections, evaluator_llm, metrics):
    """
    2단계: 준비된 데이터를 기반으로 LLM 평가 수행
    """
    all_results_dfs = []
    
    for name, data_list in ragas_data_collections.items():
        print(f"⚖️ {name} 리트리버 LLM 평가 중...")
        ragas_ds = Dataset.from_list(data_list)
        
        result = evaluate(
            dataset=ragas_ds,
            metrics=metrics,
            llm=evaluator_llm,
            run_config=RunConfig(timeout=180, max_retries=20, max_workers=10) 
        )
        
        res_df = result.to_pandas()
        res_df['retriever_name'] = name
        all_results_dfs.append(res_df)
        
    return all_results_dfs

## 결과 요약 함수

In [11]:
import pandas as pd
from tabulate import tabulate

def summarize_evaluation_results(all_results_dfs):
    """
    3단계: 모든 결과를 합치고 리트리버별 통계 계산
    """
    # 모든 결과 합치기
    total_df = pd.concat(all_results_dfs, ignore_index=True)

    # 정량 지표 요약 (평균 계산)
    summary_stats = total_df.groupby('retriever_name').agg({
        'category_alignment': 'mean',
        'vibe_relevance': 'mean'
    }).reset_index()

    # 종합 점수 계산 (가중치 적용)
    summary_stats['Total_Score'] = (summary_stats['category_alignment'] * 0.7) + (summary_stats['vibe_relevance'] * 0.3)
    summary_stats = summary_stats.sort_values(by='Total_Score', ascending=False)

    print("\n📊 리트리버별 정량 성능 비교표:")
    print(tabulate(summary_stats, headers='keys', tablefmt='psql', showindex=False))
    
    return total_df, summary_stats

## 실행

In [12]:
# 1. 검색 수행
ragas_input_data = await fetch_retrieval_results(dataset, k=5)


🚀 Loading SPLADE Model... (This happens only once)
🚀 총 70개 샘플 검색 시작 (병렬 처리)...


In [ ]:

# 2. 평가 수행
metrics = [category_alignment, vibe_relevance]
evaluated_dfs = evaluate_ragas_data(ragas_input_data, evaluator_llm, metrics)

⚖️ ensemble 리트리버 LLM 평가 중...


Evaluating: 100%|██████████| 140/140 [03:40<00:00,  1.57s/it]


⚖️ hybrid 리트리버 LLM 평가 중...


Evaluating: 100%|██████████| 140/140 [03:29<00:00,  1.50s/it]


⚖️ hyde 리트리버 LLM 평가 중...


Evaluating: 100%|██████████| 140/140 [03:37<00:00,  1.55s/it]


⚖️ dense 리트리버 LLM 평가 중...


Evaluating: 100%|██████████| 140/140 [03:50<00:00,  1.65s/it]


⚖️ sparse 리트리버 LLM 평가 중...


Evaluating:  44%|████▎     | 61/140 [01:42<01:43,  1.31s/it]

In [ ]:

# 3. 결과 요약 및 시각화
total_df, summary_stats = summarize_evaluation_results(evaluated_dfs)

# 4. 그래프 출력
import matplotlib.pyplot as plt
summary_stats.set_index('retriever_name')[['category_alignment', 'vibe_relevance', 'Total_Score']].plot(kind='bar', figsize=(10, 6))
plt.title("Retriever Performance Comparison")
plt.ylabel("Score")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()